# 실험 0: Baseline

## 0. 실행 설정 / 라이브러리

In [7]:
import os
import subprocess
import sys
import time
from pathlib import Path

RUN_SEEDS = list(range(10))
RESULTS_CSV = "exp0_baseline.csv"
CLASSIFIER_EPOCHS = 100

GPU_ID = None
if GPU_ID is None:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.used,utilization.gpu", "--format=csv,noheader,nounits"]).decode()
    rows = [tuple(int(v) for v in l.split(",")) for l in out.strip().splitlines()]
    GPU_ID = str(min(rows, key=lambda r: (r[1], r[2]))[0])
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
print("사용 GPU:", GPU_ID)

사용 GPU: 0


In [8]:
HERE = Path.cwd().resolve()
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_fscore_support
from tqdm.auto import tqdm

from models import Unet, Classifier, NoisyClassifier
from utils import set_seed, DEVICE, EMA, metrics_to_rows, append_result_rows
from diffusion import GaussianDiffusion
from data import load_am, build_imbalanced_group_split, draw_from_pool, CLASS_NAMES, TRAIN_COUNTS, TEST_COUNTS, NUM_CLASSES, IMAGE_SIZE

## 1. 데이터 / 설정

In [9]:
image_size, channels, num_classes = IMAGE_SIZE, 3, NUM_CLASSES
batch_size = 40
data_root = os.environ.get("DATA_ROOT", "../../data/AM_image_data_3split")
timesteps = 1000
CKPT_DIR = ROOT / "diffusion_train" / "checkpoints"
UNET_DIM = 32
POOL_PER_CLASS = TRAIN_COUNTS[0] - TRAIN_COUNTS[1]
MINORITY = (1, 2)
GEN_BATCH = 60

all_x, all_y, all_groups = load_am(data_root)
test_x = test_y = test_x_dev = test_y_dev = None
print(f"클래스 {CLASS_NAMES} | 전체 이미지 {all_y.bincount().tolist()} | train {TRAIN_COUNTS}, test {TEST_COUNTS}")
ddpm = GaussianDiffusion(timesteps=timesteps, image_size=image_size, channels=channels)

train_x = train_y = None


def iterate_batches(n, bs):
    perm = torch.randperm(n, device=DEVICE)
    for i in range(0, n - bs + 1, bs):
        yield perm[i:i + bs]

클래스 ['Normal', 'Underfill_50FR', 'Underfill_Fan'] | 전체 이미지 [915, 591, 459] | train [800, 80, 80], test [100, 100, 100]


## 2. 함수 정의

In [10]:
def set_split(seed):
    global train_x, train_y, test_x, test_y, test_x_dev, test_y_dev
    tr, te = build_imbalanced_group_split(all_y, all_groups, TRAIN_COUNTS, TEST_COUNTS, seed)
    assert not set(tr.tolist()) & set(te.tolist()), "train/test에 같은 이미지가 있음"
    train_x, train_y = all_x[tr].to(DEVICE), all_y[tr].to(DEVICE)
    test_x, test_y = all_x[te], all_y[te]
    test_x_dev, test_y_dev = test_x.to(DEVICE), test_y.to(DEVICE)
    assert train_y.bincount(minlength=num_classes).tolist() == TRAIN_COUNTS
    assert test_y.bincount(minlength=num_classes).tolist() == TEST_COUNTS


CLASSIFIER_LR, CLASSIFIER_BETAS = 2e-4, (0.5, 0.9)


def train_eval_classifier(seed, pool, tag):
    set_seed(seed)
    clf = Classifier(num_classes=num_classes, img_channels=channels, image_size=image_size).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=CLASSIFIER_LR, betas=CLASSIFIER_BETAS)
    loss_fn = nn.CrossEntropyLoss()
    clf.train()
    total_steps = (len(train_x) // batch_size) * CLASSIFIER_EPOCHS
    step, losses, t0 = 0, [], time.time()
    for epoch in range(CLASSIFIER_EPOCHS):
        for idx in iterate_batches(len(train_x), batch_size):
            x, y = train_x[idx], train_y[idx]
            if pool is not None:
                xf, yf = draw_from_pool(pool, y, DEVICE)
                if xf is not None:
                    x, y = torch.cat([x, xf]), torch.cat([y, yf])
            loss = loss_fn(clf(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item()); step += 1
            if step % max(1, total_steps // 5) == 0:
                print(f"[{tag} seed{seed}] step {step}/{total_steps} Cls={np.mean(losses[-50:]):.4f} elapsed={time.time()-t0:.1f}s")
    return clf


@torch.no_grad()
def evaluate_and_save(clf, seed, method, csv_path):
    clf.eval()
    pred = torch.cat([clf(test_x_dev[i:i + 500]).argmax(1) for i in range(0, len(test_y_dev), 500)]).cpu().numpy()
    y = test_y.numpy()
    cm = confusion_matrix(y, pred, labels=list(range(num_classes)))
    recall = cm.diagonal() / cm.sum(1)
    row = {"method": method, "seed": seed, "accuracy": (pred == y).mean(), "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
           **{f"recall_{n}": r for n, r in zip(CLASS_NAMES, recall)}}
    print(f"[{method} seed{seed}] acc {row['accuracy']:.4f} | macro F1 {row['macro_f1']:.4f} | recall " + ", ".join(f"{n} {r:.3f}" for n, r in zip(CLASS_NAMES, recall)))
    print(cm)
    prec, rec, f1, sup = precision_recall_fscore_support(y, pred, labels=list(range(num_classes)), zero_division=0)
    append_result_rows(metrics_to_rows({"accuracy": row["accuracy"], "precision": prec, "recall": rec, "f1": f1, "support": sup}, seed, method), csv_path)
    cpath = Path(csv_path).with_name("confusion.csv")
    pd.DataFrame([{"method": method, "seed": seed, **{f"cm_{i}{j}": int(cm[i, j]) for i in range(num_classes) for j in range(num_classes)}}]).to_csv(
        cpath, mode="a", header=not cpath.exists(), index=False)


def load_unet(mode, seed):
    path = CKPT_DIR / f"{mode}_ema_seed{seed}.pt"
    if not path.exists():
        return None
    model = Unet(dim=UNET_DIM, channels=channels, num_classes=num_classes).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

## 실험 0: Baseline & 평가

In [11]:
done = set()
if os.path.exists(RESULTS_CSV):
    _prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(_prev["method"], _prev["seed"]))

for seed in RUN_SEEDS:
    method = "0_baseline"
    if (method, seed) in done:
        print(f"[{method} seed{seed}] 결과 있음 → 건너뜀"); continue
    set_split(seed)
    print(f"\n===== Baseline | Seed {seed} | train {len(train_y)}장 =====")
    t0 = time.time()
    clf = train_eval_classifier(seed, None, tag="baseline")
    evaluate_and_save(clf, seed, method, RESULTS_CSV)
    print(f"[{method} seed{seed}] 총 {time.time()-t0:.1f}초")


===== Baseline | Seed 0 | train 960장 =====
[baseline seed0] step 480/2400 Cls=0.4268 elapsed=0.8s
[baseline seed0] step 960/2400 Cls=0.3517 elapsed=1.5s
[baseline seed0] step 1440/2400 Cls=0.2523 elapsed=2.1s
[baseline seed0] step 1920/2400 Cls=0.2007 elapsed=2.7s
[baseline seed0] step 2400/2400 Cls=0.1539 elapsed=3.3s
[0_baseline seed0] acc 0.7267 | macro F1 0.6976 | recall Normal 1.000, Underfill_50FR 0.290, Underfill_Fan 0.890
[[100   0   0]
 [ 68  29   3]
 [ 10   1  89]]
[0_baseline seed0] 총 3.8초

===== Baseline | Seed 1 | train 960장 =====
[baseline seed1] step 480/2400 Cls=0.4170 elapsed=0.6s
[baseline seed1] step 960/2400 Cls=0.3068 elapsed=1.3s
[baseline seed1] step 1440/2400 Cls=0.2293 elapsed=1.9s
[baseline seed1] step 1920/2400 Cls=0.1886 elapsed=2.5s
[baseline seed1] step 2400/2400 Cls=0.1544 elapsed=3.1s
[0_baseline seed1] acc 0.6900 | macro F1 0.6437 | recall Normal 0.990, Underfill_50FR 0.190, Underfill_Fan 0.890
[[99  0  1]
 [79 19  2]
 [ 9  2 89]]
[0_baseline seed1] 총 

## 결과 요약 / 대응표본 t-test

In [12]:
from scipy import stats

ALL_CSVS = {
    "exp0 baseline": ROOT / "exp0_baseline" / "exp0_baseline.csv",
    "exp1 오버샘플링": ROOT / "exp1_oversampling" / "exp1_oversampling.csv",
    "exp2 guidance": ROOT / "exp2_classifier_guidance" / "exp2_classifier_guidance.csv",
}

runs, own = {}, set()
for name, path in ALL_CSVS.items():
    if Path(path).exists() and Path(path).stat().st_size > 0:
        df = pd.read_csv(path).drop_duplicates(["method", "seed", "class"], keep="last")
        df["class"] = df["class"].astype(str)
        for method, dm in df.groupby("method"):
            macro = dm[dm["class"] == "macro"].set_index("seed")
            wide = pd.DataFrame({"accuracy": macro["accuracy"], "macro_f1": macro["f1"]})
            for i, n in enumerate(CLASS_NAMES):
                wide[f"recall_{n}"] = dm[dm["class"] == str(i)].set_index("seed")["recall"]
            key = name if df["method"].nunique() == 1 else f"{name} [{method.split('step10_')[-1]}]"
            runs[key] = wide
            if Path(path).name == Path(RESULTS_CSV).name:
                own.add(key)

if not own:
    print("아직 결과가 없다")
else:
    seeds = sorted(set.intersection(*[set(runs[k].index) for k in own]))
    rows = []
    for name, g in runs.items():
        g = g.reindex(seeds).dropna(subset=["accuracy"])
        rows.append({"방법": name, "n": len(g), "Accuracy": f"{g.accuracy.mean():.4f}", "macro F1": f"{g.macro_f1.mean():.4f}",
                     **{f"recall {n}": f"{g[f'recall_{n}'].mean():.3f}" for n in CLASS_NAMES}})
    print(f"===== 성능 (이 노트북의 seed {seeds}) =====")
    display(pd.DataFrame(rows).set_index("방법"))

    rows = []
    for k in sorted(own):
        for name, ref in runs.items():
            if name in own:
                continue
            common = runs[k].index.intersection(ref.index)
            if len(common) < 2:
                continue
            for metric in ("accuracy", "macro_f1"):
                d = runs[k].loc[common, metric] - ref.loc[common, metric]
                t, p = stats.ttest_rel(runs[k].loc[common, metric], ref.loc[common, metric])
                rows.append({"이 노트북": k, "비교 대상": name, "지표": metric, "n": len(common), "평균 차이": round(d.mean(), 4), "p": round(p, 4), "우세 seed": f"{(d > 0).sum()}/{len(d)}"})
    print("\n===== 대응표본 t-test (이 노트북 − 비교 대상) =====")
    display(pd.DataFrame(rows).set_index(["이 노트북", "비교 대상", "지표"]) if rows else pd.DataFrame())

===== 성능 (이 노트북의 seed [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]) =====


,n,Accuracy,macro F1,recall Normal,recall Underfill_50FR,recall Underfill_Fan
방법,,,,,,
exp0 baseline,10,0.7520,0.7261,0.984,0.350,0.922
exp1 오버샘플링,10,0.7873,0.7755,0.925,0.533,0.904
exp2 guidance,10,0.8060,0.7966,0.947,0.556,0.915



===== 대응표본 t-test (이 노트북 − 비교 대상) =====


n   평균 차이       p 우세 seed
이 노트북         비교 대상         지표                                  
exp0 baseline exp1 오버샘플링    accuracy  10 -0.0353  0.1448    3/10
                            macro_f1  10 -0.0494  0.1065    3/10
              exp2 guidance accuracy  10 -0.0540  0.0222    2/10
                            macro_f1  10 -0.0705  0.0178    2/10